In [3]:
# =============================================================================
# SECTION — Broker Metrics from Prometheus
# =============================================================================

import requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import glob
import os


PROM_URL = "http://localhost:9090"  
INPUT_DIR = Path("../results/20260507_233056_balanced/consumer/consumer-sts-*_metrics/2026-05-07-23-06/")
OUTPUT_DIR = Path("./balanced_analysis_output")
BROKER_OUTPUT_DIR = OUTPUT_DIR / "broker_metrics"
BROKER_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)



In [4]:
def read_many_csv(pattern):
    files = sorted(glob.glob(str(pattern), recursive=True))

    if not files:
        print(f"[WARN] No files found for: {pattern}")
        return pd.DataFrame()

    dfs = []

    for f in files:
        try:
            df = pd.read_csv(f)
            df["source_file"] = os.path.basename(f)
            dfs.append(df)
        except Exception as e:
            print(f"[WARN] Failed reading {f}: {e}")

    return pd.concat(dfs, ignore_index=True)

def q(series, p):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return float(s.quantile(p))

def coefficient_of_variation(series):
    s = pd.to_numeric(series, errors="coerce").dropna()

    if len(s) == 0:
        return np.nan

    mean = s.mean()

    if mean == 0:
        return np.nan

    return float(s.std() / mean)

def add_time_bucket(df, ts_col, bucket_sec=5):

    out = df.copy()

    out[ts_col] = pd.to_numeric(out[ts_col], errors="coerce")

    out = out.dropna(subset=[ts_col])

    out["bucket"] = (
        (out[ts_col] // bucket_sec).astype(int) * bucket_sec
    )

    out["time_rel_sec"] = (
        out["bucket"] - out["bucket"].min()
    )

    return out


In [5]:
metrics_df = read_many_csv(
    INPUT_DIR / "**" / "consumer_metrics_*.csv"
)

lag_df = read_many_csv(
    INPUT_DIR / "**" / "consumer_partition_lag_*.csv"
)

per_msg_df = read_many_csv(
    INPUT_DIR / "**" / "per_message_latency_*.csv"
)

rebalance_df = read_many_csv(
    INPUT_DIR / "**" / "consumer_rebalance_events_*.csv"
)

print("metrics_df:", metrics_df.shape)
print("lag_df:", lag_df.shape)
print("per_msg_df:", per_msg_df.shape)
print("rebalance_df:", rebalance_df.shape)


metrics_df: (264, 21)
lag_df: (38722, 9)
per_msg_df: (54614, 9)
rebalance_df: (47, 7)


In [6]:
def prom_query_range(query, start, end, step="5s"):
    """
    Query Prometheus range API and return a dataframe.
    start/end should be Unix timestamps.
    """
    url = f"{PROM_URL}/api/v1/query_range"

    params = {
        "query": query,
        "start": start,
        "end": end,
        "step": step,
    }

    r = requests.get(url, params=params)
    r.raise_for_status()

    data = r.json()["data"]["result"]

    rows = []

    for series in data:
        metric = series["metric"]
        values = series["values"]

        for ts, value in values:
            row = {
                "ts": float(ts),
                "value": float(value),
            }

            row.update(metric)
            rows.append(row)

    return pd.DataFrame(rows)
    
start_ts = per_msg_df["recv_ts"].min()
end_ts   = per_msg_df["recv_ts"].max()

print(start_ts, end_ts)

broker_queries = {
    "request_queue_size": 'kafka_network_requestmetrics_requestqueuesize',
    "response_queue_size": 'kafka_network_requestmetrics_responsequeuesize',
    "request_handler_idle": 'kafka_server_kafkarequesthandlerpool_requesthandleravgidlepercent',
    "network_processor_idle": 'kafka_network_socketserver_networkprocessoravgidlepercent',
    "produce_request_time_mean": 'kafka_network_requestmetrics_totaltimems_mean{request="Produce"}',
    "fetch_consumer_time_mean": 'kafka_network_requestmetrics_totaltimems_mean{request="FetchConsumer"}',
    "bytes_in_rate": 'rate(kafka_server_brokertopicmetrics_bytesin_total[30s])',
    "bytes_out_rate": 'rate(kafka_server_brokertopicmetrics_bytesout_total[30s])',
    "messages_in_rate": 'rate(kafka_server_brokertopicmetrics_messagesin_total[30s])',
}

broker_metric_dfs = {}

for name, query in broker_queries.items():
    try:
        df = prom_query_range(
            query=query,
            start=start_ts,
            end=end_ts,
            step="5s"
        )

        broker_metric_dfs[name] = df

        df.to_csv(
            BROKER_OUTPUT_DIR / f"{name}.csv",
            index=False
        )

        print(f"[OK] {name}: {df.shape}")

    except Exception as e:
        print(f"[WARN] Failed {name}: {e}")

1778216814.1983318 1778218105.1639175
[WARN] Failed request_queue_size: HTTPConnectionPool(host='localhost', port=9090): Max retries exceeded with url: /api/v1/query_range?query=kafka_network_requestmetrics_requestqueuesize&start=1778216814.1983318&end=1778218105.1639175&step=5s (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x125623bb0>: Failed to establish a new connection: [Errno 61] Connection refused'))
[WARN] Failed response_queue_size: HTTPConnectionPool(host='localhost', port=9090): Max retries exceeded with url: /api/v1/query_range?query=kafka_network_requestmetrics_responsequeuesize&start=1778216814.1983318&end=1778218105.1639175&step=5s (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1256e6880>: Failed to establish a new connection: [Errno 61] Connection refused'))
[WARN] Failed request_handler_idle: HTTPConnectionPool(host='localhost', port=9090): Max retries exceeded with url: /api/v1/query_range?query=kafka_server

In [8]:
def plot_broker_metric(name):
    df = broker_metric_dfs.get(name)

    if df is None or df.empty:
        print(f"No data for {name}")
        return

    plt.figure(figsize=(12, 5))

    label_col = "pod" if "pod" in df.columns else "instance"

    for label, g in df.groupby(label_col):
        g = g.sort_values("ts")
        plt.plot(
            g["ts"] - g["ts"].min(),
            g["value"],
            label=label,
            linewidth=1
        )

    plt.xlabel("Time relative to metric start (sec)")
    plt.ylabel(name)
    plt.title(f"Broker metric: {name}")
    plt.legend(fontsize=8)
    plt.grid(True)

    plt.savefig(
        BROKER_OUTPUT_DIR / f"{name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [9]:
plot_broker_metric("request_queue_size")
plot_broker_metric("response_queue_size")
plot_broker_metric("produce_request_time_mean")
plot_broker_metric("fetch_consumer_time_mean")
plot_broker_metric("request_handler_idle")
plot_broker_metric("network_processor_idle")

No data for request_queue_size
No data for response_queue_size
No data for produce_request_time_mean
No data for fetch_consumer_time_mean
No data for request_handler_idle
No data for network_processor_idle
